# Advanced Gradient Boosting: XGBoost & LightGBM for Complex Environmental Data

## Learning Objectives
By the end of this notebook, you will:
- Understand when and why to use gradient boosting for structured/tabular data
- Master XGBoost and LightGBM for complex environmental predictions
- Learn to capture non-linear interactions in coastal/environmental datasets
- Interpret feature importance for scientific insights
- Apply these methods to real-world coastal response modeling

## Why Gradient Boosting for Environmental Data?

### Key Advantages:
1. **Excellent for Structured/Tabular Data**: Environmental datasets are often tabular with mixed data types
2. **Complex Non-Linear Interactions**: Can capture relationships between elevation, sea-level rise, temperature, etc.
3. **Feature Importance**: Provides explicit metrics showing key drivers of environmental response
4. **Handles Missing Data**: Common in environmental monitoring datasets
5. **High Predictive Accuracy**: Often wins competitions on structured data

### Best Suited When:
- Dataset is diverse with complex variable interactions
- High predictive accuracy is essential
- You need to understand feature contributions
- Working with mixed data types (numerical, categorical)


In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Install required packages (uncomment if needed)
# !pip install xgboost lightgbm shap

import xgboost as xgb
import lightgbm as lgb
import shap

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")

## 1. Creating a Coastal Response Dataset

Let's simulate a realistic coastal response dataset with complex interactions:

In [ ]:
# Create synthetic coastal response dataset
np.random.seed(42)
n_samples = 5000

# Generate features
data = {
    # Elevation features
    'elevation_m': np.random.uniform(0, 50, n_samples),
    'slope_degrees': np.random.uniform(0, 45, n_samples),
    'distance_to_coast_km': np.random.uniform(0, 20, n_samples),
    
    # Environmental features
    'sea_level_rise_mm_yr': np.random.uniform(1, 8, n_samples),
    'wave_height_m': np.random.uniform(0.5, 4.0, n_samples),
    'storm_frequency': np.random.poisson(2, n_samples),
    'temperature_change_c': np.random.uniform(-1, 5, n_samples),
    
    # Geological features
    'soil_permeability': np.random.uniform(0.01, 0.9, n_samples),
    'bedrock_depth_m': np.random.uniform(1, 100, n_samples),
    
    # Human impact features
    'population_density': np.random.uniform(0, 1000, n_samples),
    'infrastructure_density': np.random.uniform(0, 1, n_samples)
}

# Add categorical features
data['coastal_type'] = np.random.choice(['rocky', 'sandy', 'muddy', 'mixed'], n_samples)
data['protection_level'] = np.random.choice(['none', 'low', 'medium', 'high'], n_samples)

# Create DataFrame
df = pd.DataFrame(data)

# Create complex target variable with non-linear interactions
def calculate_coastal_vulnerability(row):
    """Calculate coastal vulnerability with complex interactions."""
    
    # Base vulnerability from elevation and sea level rise interaction
    base_vuln = (row['sea_level_rise_mm_yr'] ** 1.5) / (row['elevation_m'] + 1) ** 0.8
    
    # Wave and storm interaction
    wave_storm_effect = row['wave_height_m'] * (row['storm_frequency'] + 1) ** 0.7
    
    # Slope protection effect (non-linear)
    slope_effect = 1 / (1 + np.exp(-0.1 * (row['slope_degrees'] - 20)))
    
    # Soil and distance interaction
    soil_distance_effect = row['soil_permeability'] * np.log(row['distance_to_coast_km'] + 1)
    
    # Temperature amplification
    temp_amplifier = 1 + 0.2 * row['temperature_change_c']
    
    # Coastal type modifier
    type_modifier = {'rocky': 0.7, 'sandy': 1.2, 'muddy': 1.5, 'mixed': 1.0}[row['coastal_type']]
    
    # Protection effect
    protection_effect = {'none': 1.0, 'low': 0.8, 'medium': 0.6, 'high': 0.3}[row['protection_level']]
    
    # Human impact amplifier
    human_amplifier = 1 + 0.001 * row['population_density'] + 0.3 * row['infrastructure_density']
    
    # Combine all effects
    vulnerability = (
        base_vuln * wave_storm_effect * slope_effect * 
        temp_amplifier * type_modifier * protection_effect * 
        human_amplifier * (1 - soil_distance_effect)
    )
    
    return max(0, vulnerability)  # Ensure non-negative

# Apply the function to create target
df['vulnerability_index'] = df.apply(calculate_coastal_vulnerability, axis=1)

# Add some noise
df['vulnerability_index'] += np.random.normal(0, 0.1 * df['vulnerability_index'].std(), n_samples)

print(f"Dataset created with {len(df)} samples and {len(df.columns)-1} features")
print(f"Target variable (vulnerability_index) range: {df['vulnerability_index'].min():.2f} to {df['vulnerability_index'].max():.2f}")

# Display basic info
df.head()

In [ ]:
# Explore the dataset
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution of target variable
axes[0, 0].hist(df['vulnerability_index'], bins=50, alpha=0.7, color='skyblue')
axes[0, 0].set_title('Distribution of Coastal Vulnerability Index')
axes[0, 0].set_xlabel('Vulnerability Index')
axes[0, 0].set_ylabel('Frequency')

# Key relationships
axes[0, 1].scatter(df['elevation_m'], df['vulnerability_index'], alpha=0.5, s=1)
axes[0, 1].set_title('Elevation vs Vulnerability')
axes[0, 1].set_xlabel('Elevation (m)')
axes[0, 1].set_ylabel('Vulnerability Index')

axes[1, 0].scatter(df['sea_level_rise_mm_yr'], df['vulnerability_index'], alpha=0.5, s=1)
axes[1, 0].set_title('Sea Level Rise vs Vulnerability')
axes[1, 0].set_xlabel('Sea Level Rise (mm/yr)')
axes[1, 0].set_ylabel('Vulnerability Index')

# Categorical relationships
df.boxplot(column='vulnerability_index', by='coastal_type', ax=axes[1, 1])
axes[1, 1].set_title('Vulnerability by Coastal Type')
axes[1, 1].set_xlabel('Coastal Type')

plt.tight_layout()
plt.show()

# Correlation matrix for numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Environmental Features')
plt.tight_layout()
plt.show()

## 2. Data Preprocessing for Gradient Boosting

Unlike neural networks, gradient boosting doesn't require feature scaling, but we need to handle categorical variables:

In [ ]:
# Prepare features and target
X = df.drop('vulnerability_index', axis=1)
y = df['vulnerability_index']

# Handle categorical variables
categorical_features = ['coastal_type', 'protection_level']
numerical_features = [col for col in X.columns if col not in categorical_features]

print(f"Categorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

# For XGBoost, we'll use label encoding
X_encoded = X.copy()
label_encoders = {}

for cat_col in categorical_features:
    le = LabelEncoder()
    X_encoded[cat_col] = le.fit_transform(X_encoded[cat_col])
    label_encoders[cat_col] = le
    print(f"{cat_col} encoded: {le.classes_}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 3. XGBoost Implementation

XGBoost is excellent for capturing complex interactions in environmental data:

In [ ]:
# Basic XGBoost model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_pred = xgb_model.predict(X_test)

# Evaluate performance
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Performance:")
print(f"RMSE: {xgb_rmse:.4f}")
print(f"MAE: {xgb_mae:.4f}")
print(f"R² Score: {xgb_r2:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, xgb_pred, alpha=0.6, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Vulnerability Index')
plt.ylabel('Predicted Vulnerability Index')
plt.title(f'XGBoost: Predictions vs Actual (R² = {xgb_r2:.3f})')
plt.tight_layout()
plt.show()

## 4. LightGBM Implementation

LightGBM often provides better performance and faster training:

In [ ]:
# LightGBM can handle categorical features natively
X_lgb = X.copy()

# Convert categorical columns to 'category' dtype for LightGBM
for cat_col in categorical_features:
    X_lgb[cat_col] = X_lgb[cat_col].astype('category')

# Split data for LightGBM
X_train_lgb, X_test_lgb, y_train_lgb, y_test_lgb = train_test_split(
    X_lgb, y, test_size=0.2, random_state=42
)

# Create LightGBM model
lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1  # Suppress output
)

# Train the model
lgb_model.fit(
    X_train_lgb, y_train_lgb,
    categorical_feature=categorical_features
)

# Make predictions
lgb_pred = lgb_model.predict(X_test_lgb)

# Evaluate performance
lgb_mse = mean_squared_error(y_test_lgb, lgb_pred)
lgb_rmse = np.sqrt(lgb_mse)
lgb_mae = mean_absolute_error(y_test_lgb, lgb_pred)
lgb_r2 = r2_score(y_test_lgb, lgb_pred)

print("LightGBM Performance:")
print(f"RMSE: {lgb_rmse:.4f}")
print(f"MAE: {lgb_mae:.4f}")
print(f"R² Score: {lgb_r2:.4f}")

# Compare models
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# XGBoost predictions
axes[0].scatter(y_test, xgb_pred, alpha=0.6, s=10, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Vulnerability Index')
axes[0].set_ylabel('Predicted Vulnerability Index')
axes[0].set_title(f'XGBoost (R² = {xgb_r2:.3f})')

# LightGBM predictions
axes[1].scatter(y_test_lgb, lgb_pred, alpha=0.6, s=10, color='green')
axes[1].plot([y_test_lgb.min(), y_test_lgb.max()], [y_test_lgb.min(), y_test_lgb.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Vulnerability Index')
axes[1].set_ylabel('Predicted Vulnerability Index')
axes[1].set_title(f'LightGBM (R² = {lgb_r2:.3f})')

plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis

One of the key advantages of gradient boosting is interpretable feature importance:

In [ ]:
# Get feature importance from both models
xgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

lgb_importance = pd.DataFrame({
    'feature': X_train_lgb.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost feature importance
axes[0].barh(range(len(xgb_importance)), xgb_importance['importance'])
axes[0].set_yticks(range(len(xgb_importance)))
axes[0].set_yticklabels(xgb_importance['feature'])
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('XGBoost Feature Importance')
axes[0].invert_yaxis()

# LightGBM feature importance
axes[1].barh(range(len(lgb_importance)), lgb_importance['importance'])
axes[1].set_yticks(range(len(lgb_importance)))
axes[1].set_yticklabels(lgb_importance['feature'])
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('LightGBM Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Top 5 most important features (XGBoost):")
for i, row in xgb_importance.head().iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

print("\nTop 5 most important features (LightGBM):")
for i, row in lgb_importance.head().iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

## 6. Hyperparameter Optimization

Fine-tuning gradient boosting models for optimal performance:

In [ ]:
# Hyperparameter tuning for XGBoost
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 0.9, 1.0]
}

# Note: Using a subset of parameters for demonstration
# In practice, consider using RandomizedSearchCV for efficiency
xgb_simplified_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 9],
    'learning_rate': [0.1, 0.2]
}

xgb_grid_search = GridSearchCV(
    xgb.XGBRegressor(random_state=42),
    xgb_simplified_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Tuning XGBoost hyperparameters...")
xgb_grid_search.fit(X_train, y_train)

print(f"\nBest XGBoost parameters: {xgb_grid_search.best_params_}")
print(f"Best cross-validation score: {-xgb_grid_search.best_score_:.4f}")

# Train best model
best_xgb = xgb_grid_search.best_estimator_
best_xgb_pred = best_xgb.predict(X_test)
best_xgb_r2 = r2_score(y_test, best_xgb_pred)

print(f"Optimized XGBoost R² score: {best_xgb_r2:.4f}")
print(f"Improvement over default: {best_xgb_r2 - xgb_r2:.4f}")

## 7. SHAP Values for Advanced Interpretability

SHAP (SHapley Additive exPlanations) provides detailed feature contribution analysis:

In [ ]:
# Create SHAP explainer for the best XGBoost model
explainer = shap.Explainer(best_xgb)

# Calculate SHAP values for a subset of test data (for performance)
shap_values = explainer(X_test[:1000])

# Summary plot
plt.figure(figsize=(12, 8))
shap.plots.beeswarm(shap_values, max_display=10)
plt.title('SHAP Feature Importance for Coastal Vulnerability Prediction')
plt.tight_layout()
plt.show()

# Waterfall plot for a single prediction
plt.figure(figsize=(12, 8))
shap.plots.waterfall(shap_values[0])
plt.title('SHAP Waterfall Plot - Individual Prediction Explanation')
plt.tight_layout()
plt.show()

# Feature interaction analysis
print("\nAnalyzing feature interactions...")
interaction_values = shap.Explainer(best_xgb).shap_interaction_values(X_test[:500])

# Plot interaction between top 2 features
top_features = xgb_importance.head(2)['feature'].values
if len(top_features) >= 2:
    feature1_idx = list(X_test.columns).index(top_features[0])
    feature2_idx = list(X_test.columns).index(top_features[1])
    
    plt.figure(figsize=(10, 6))
    shap.plots.scatter(
        shap_values[:500, feature1_idx], 
        color=shap_values[:500, feature2_idx]
    )
    plt.title(f'Feature Interaction: {top_features[0]} vs {top_features[1]}')
    plt.show()

## 8. Capturing Complex Non-Linear Interactions

Let's explore specific interactions relevant to coastal vulnerability:

In [ ]:
# Analyze elevation-sea level rise interaction
test_data = X_test.copy()
test_data['predicted_vulnerability'] = best_xgb_pred
test_data['actual_vulnerability'] = y_test.values

# Create interaction plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Elevation vs Sea Level Rise interaction
scatter = axes[0, 0].scatter(
    test_data['elevation_m'], 
    test_data['sea_level_rise_mm_yr'],
    c=test_data['predicted_vulnerability'],
    cmap='RdYlBu_r',
    alpha=0.7,
    s=20
)
axes[0, 0].set_xlabel('Elevation (m)')
axes[0, 0].set_ylabel('Sea Level Rise (mm/yr)')
axes[0, 0].set_title('Predicted Vulnerability: Elevation vs SLR')
plt.colorbar(scatter, ax=axes[0, 0], label='Vulnerability')

# Wave height vs storm frequency interaction
scatter2 = axes[0, 1].scatter(
    test_data['wave_height_m'],
    test_data['storm_frequency'],
    c=test_data['predicted_vulnerability'],
    cmap='RdYlBu_r',
    alpha=0.7,
    s=20
)
axes[0, 1].set_xlabel('Wave Height (m)')
axes[0, 1].set_ylabel('Storm Frequency')
axes[0, 1].set_title('Predicted Vulnerability: Wave vs Storms')
plt.colorbar(scatter2, ax=axes[0, 1], label='Vulnerability')

# Protection level effectiveness
protection_effect = test_data.groupby('protection_level').agg({
    'predicted_vulnerability': ['mean', 'std'],
    'actual_vulnerability': ['mean', 'std']
markdown
gb-1
markdown
# Gradient Boosting (GBM) — A Gentle Intro

## What it is
Gradient Boosting builds an ensemble of small decision trees, each one correcting errors of the previous model (sequential boosting).

## What it does
- Learns complex non-linear patterns by adding trees step-by-step (additive model).
- Balances bias/variance via `learning_rate`, number of trees, and tree depth.

## When to use
- Tabular data with mixed signals (linear + non-linear).
- You want strong performance with interpretable feature importance.
code
gb-2
python
# Imports
import numpy as np  # numerical arrays
import matplotlib.pyplot as plt  # plotting
from sklearn.datasets import make_regression  # synthetic regression data
from sklearn.model_selection import train_test_split  # split data into train/test
from sklearn.metrics import mean_squared_error, r2_score  # evaluation metrics
from sklearn.ensemble import GradientBoostingRegressor  # scikit-learn GBM regressor
code
gb-3
python
# Create a simple non-linear regression dataset
X, y = make_regression(n_samples=500, n_features=5, noise=10.0, random_state=42)  # base linear data
y = y + 15*np.sin(X[:, 0]) - 10*np.cos(X[:, 1])  # inject non-linear signal into two features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # train/test split
code
gb-4
python
# Define and train a compact Gradient Boosting model
model = GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=3, random_state=42)  # small trees + step size
model.fit(X_train, y_train)  # fit additive boosted trees
code
gb-5
python
# Evaluate predictive performance
y_pred = model.predict(X_test)  # predict on holdout set
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))  # root mean squared error
r2 = float(r2_score(y_test, y_pred))  # proportion of variance explained
print(f'RMSE: {rmse:.3f}, R²: {r2:.3f}')  # basic metrics
code
gb-6
python
# Explain the model via feature importance
importances = model.feature_importances_  # relative contribution from each feature
plt.bar(range(len(importances)), importances)  # bar chart of importances
plt.xlabel('Feature index')  # x-axis label
plt.ylabel('Importance')  # y-axis label
plt.title('Gradient Boosting Feature Importance')  # title
plt.show()  # render figure
markdown
gb-7
markdown
### Key ideas
- Boosting builds models sequentially to fix prior errors.
- Small trees (weak learners) plus a learning rate control bias/variance.
- Tune `n_estimators`, `learning_rate`, and `max_depth` for performance vs overfitting.
4
5
axes[1, 0].legend()

# Coastal type vulnerability distribution
test_data.boxplot(column='predicted_vulnerability', by='coastal_type', ax=axes[1, 1])
axes[1, 1].set_title('Vulnerability by Coastal Type')
axes[1, 1].set_xlabel('Coastal Type')
axes[1, 1].set_ylabel('Predicted Vulnerability')

plt.tight_layout()
plt.show()

# Print insights
print("Key Insights from Gradient Boosting Analysis:")
print("=" * 50)
print(f"1. Most important feature: {xgb_importance.iloc[0]['feature']}")
print(f"2. Model captures {best_xgb_r2:.1%} of variance in coastal vulnerability")
print(f"3. Protection effectiveness ranking:")
for level, vuln in zip(protection_levels, pred_means):
    print(f"   - {level}: {vuln:.2f} avg vulnerability")


## 9. Model Validation and Cross-Validation

Robust validation for environmental predictions:

In [ ]:
# Cross-validation with multiple metrics
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer

# Define custom scoring functions
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Scoring metrics
scoring = {
    'r2': 'r2',
    'neg_mse': 'neg_mean_squared_error',
    'neg_mae': 'neg_mean_absolute_error',
    'mape': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

# Perform cross-validation
cv_results = cross_validate(
    best_xgb, X_train, y_train,
    cv=5,
    scoring=scoring,
    return_train_score=True
)

# Display results
print("Cross-Validation Results (5-fold):")
print("=" * 40)
for metric in ['r2', 'neg_mse', 'neg_mae', 'mape']:
    test_scores = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    
    if metric.startswith('neg_'):
        test_scores = -test_scores
        train_scores = -train_scores
        metric_name = metric[4:].upper()
    else:
        metric_name = metric.upper()
    
    print(f"{metric_name}:")
    print(f"  Test: {test_scores.mean():.4f} (+/- {test_scores.std() * 2:.4f})")
    print(f"  Train: {train_scores.mean():.4f} (+/- {train_scores.std() * 2:.4f})")
    print(f"  Overfitting check: {abs(train_scores.mean() - test_scores.mean()):.4f}")
    print()

# Learning curve
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_xgb, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=3,
    scoring='r2',
    n_jobs=-1
)

# Plot learning curve
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training Score')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation Score')
plt.fill_between(train_sizes, 
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), 
                alpha=0.1)
plt.fill_between(train_sizes, 
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), 
                alpha=0.1)
plt.xlabel('Training Set Size')
plt.ylabel('R² Score')
plt.title('Learning Curve: XGBoost for Coastal Vulnerability')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Practical Application: Scenario Analysis

Use the trained model to analyze different climate scenarios:

In [ ]:
# Create scenario analysis for policy decisions
def create_scenario_data(base_data, scenario_name, modifications):
    """Create modified dataset for scenario analysis."""
    scenario_data = base_data.copy()
    
    for feature, change in modifications.items():
        if isinstance(change, dict):
            if change['type'] == 'multiply':
                scenario_data[feature] *= change['factor']
            elif change['type'] == 'add':
                scenario_data[feature] += change['value']
        else:
            scenario_data[feature] = change
    
    return scenario_data

# Define climate scenarios
scenarios = {
    'Current (Baseline)': {},
    'Moderate Climate Change': {
        'sea_level_rise_mm_yr': {'type': 'multiply', 'factor': 1.5},
        'temperature_change_c': {'type': 'add', 'value': 2.0},
        'storm_frequency': {'type': 'multiply', 'factor': 1.3}
    },
    'Severe Climate Change': {
        'sea_level_rise_mm_yr': {'type': 'multiply', 'factor': 2.5},
        'temperature_change_c': {'type': 'add', 'value': 4.0},
        'storm_frequency': {'type': 'multiply', 'factor': 2.0}
    },
    'Enhanced Protection': {
        'protection_level': 'high'  # Set all to high protection
    },
    'Severe Change + Protection': {
        'sea_level_rise_mm_yr': {'type': 'multiply', 'factor': 2.5},
        'temperature_change_c': {'type': 'add', 'value': 4.0},
        'storm_frequency': {'type': 'multiply', 'factor': 2.0},
        'protection_level': 'high'
    }
}

# Analyze scenarios
scenario_results = {}
sample_data = X_test.iloc[:1000].copy()  # Use subset for analysis

for scenario_name, modifications in scenarios.items():
    # Create scenario data
    scenario_data = create_scenario_data(sample_data, scenario_name, modifications)
    
    # Handle categorical encoding if needed
    scenario_encoded = scenario_data.copy()
    for cat_col in categorical_features:
        if cat_col in scenario_encoded.columns:
            if scenario_encoded[cat_col].dtype == 'object':
                # Apply same encoding as training
                scenario_encoded[cat_col] = label_encoders[cat_col].transform(
                    scenario_encoded[cat_col]
                )
    
    # Predict vulnerability
    predictions = best_xgb.predict(scenario_encoded)
    
    scenario_results[scenario_name] = {
        'mean_vulnerability': predictions.mean(),
        'std_vulnerability': predictions.std(),
        'high_risk_percent': (predictions > predictions.quantile(0.8)).mean() * 100,
        'predictions': predictions
    }

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Mean vulnerability comparison
scenario_names = list(scenario_results.keys())
mean_vulns = [scenario_results[name]['mean_vulnerability'] for name in scenario_names]
std_vulns = [scenario_results[name]['std_vulnerability'] for name in scenario_names]

axes[0, 0].bar(range(len(scenario_names)), mean_vulns, yerr=std_vulns, 
              capsize=5, alpha=0.7, color=['blue', 'orange', 'red', 'green', 'purple'])
axes[0, 0].set_xticks(range(len(scenario_names)))
axes[0, 0].set_xticklabels(scenario_names, rotation=45, ha='right')
axes[0, 0].set_ylabel('Mean Vulnerability Index')
axes[0, 0].set_title('Average Vulnerability by Scenario')

# High-risk percentage comparison
high_risk_pcts = [scenario_results[name]['high_risk_percent'] for name in scenario_names]
axes[0, 1].bar(range(len(scenario_names)), high_risk_pcts, alpha=0.7,
              color=['blue', 'orange', 'red', 'green', 'purple'])
axes[0, 1].set_xticks(range(len(scenario_names)))
axes[0, 1].set_xticklabels(scenario_names, rotation=45, ha='right')
axes[0, 1].set_ylabel('High Risk Areas (%)')
axes[0, 1].set_title('Percentage of High-Risk Areas by Scenario')

# Distribution comparison
for i, (name, results) in enumerate(scenario_results.items()):
    axes[1, 0].hist(results['predictions'], bins=30, alpha=0.5, 
                   label=name, density=True)
axes[1, 0].set_xlabel('Vulnerability Index')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Vulnerability Distribution by Scenario')
axes[1, 0].legend()

# Risk level changes
baseline_mean = scenario_results['Current (Baseline)']['mean_vulnerability']
risk_changes = [(mean_vulns[i] - baseline_mean) / baseline_mean * 100 
               for i in range(len(scenario_names))]

colors = ['gray' if change == 0 else 'red' if change > 0 else 'green' 
         for change in risk_changes]
axes[1, 1].bar(range(len(scenario_names)), risk_changes, alpha=0.7, color=colors)
axes[1, 1].set_xticks(range(len(scenario_names)))
axes[1, 1].set_xticklabels(scenario_names, rotation=45, ha='right')
axes[1, 1].set_ylabel('Change from Baseline (%)')
axes[1, 1].set_title('Vulnerability Change Relative to Baseline')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Print quantitative results
print("\nScenario Analysis Results:")
print("=" * 50)
for name, results in scenario_results.items():
    print(f"\n{name}:")
    print(f"  Mean Vulnerability: {results['mean_vulnerability']:.3f}")
    print(f"  High-Risk Areas: {results['high_risk_percent']:.1f}%")
    if name != 'Current (Baseline)':
        change = (results['mean_vulnerability'] - baseline_mean) / baseline_mean * 100
        print(f"  Change from Baseline: {change:+.1f}%")

## Key Takeaways

### When to Use Gradient Boosting (XGBoost/LightGBM):

✅ **Perfect for:**
- Structured/tabular environmental data
- Complex non-linear interactions (elevation × sea-level rise)
- High predictive accuracy requirements
- Feature importance interpretation
- Mixed data types (numerical + categorical)

### Advantages Demonstrated:
1. **Captures Complex Interactions**: Elevation-SLR, wave-storm combinations
2. **Interpretable Results**: Feature importance + SHAP analysis
3. **Robust Performance**: High R² scores with proper validation
4. **Scenario Analysis**: Enables policy decision support

### Best Practices:
- Use proper cross-validation
- Tune hyperparameters systematically
- Validate with SHAP for interpretability
- Apply to scenario planning
- Monitor for overfitting

Gradient boosting excels when you need both high accuracy and interpretability for complex environmental predictions!